In [ ]:
pip install pyspark

In [ ]:
%%writefile stockmovements.csv
product_id,product_name,warehouse_id,action,quantity,date
1,Laptop,1,STOCK_IN,25,2026-10-01
2,Headphones,1,STOCK_IN,15,2026-10-02
1,Laptop,1,STOCK_ADJUSTMENT,-5,2026-10-03
3,Office Chair,2,STOCK_IN,8,2026-10-04
2,Headphones,1,STOCK_ADJUSTMENT,-10,2026-10-05
1,Laptop,2,STOCK_IN,40,2026-10-06

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, when
)

spark = SparkSession.builder \
    .appName("InventoryWarehouseAnalysis") \
    .master("local[*]") \
    .getOrCreate()
df = spark.read.csv(
    "stockmovements.csv",
    header=True,
    inferSchema=True
)
df = df.dropna(
    subset=["product_id", "warehouse_id", "quantity"]
)
stock = df.groupBy(
    "warehouse_id", "product_id", "product_name"
).agg(
    spark_sum("quantity").alias("current_stock")
)
stock = stock.withColumn(
    "stock_status",
    when(col("current_stock") < 10, "Understocked")
    .when(col("current_stock") > 50, "Overstocked")
    .otherwise("Normal")
)

print("Product stock by warehouse:")
stock.show()
warehouse_summary = stock.groupBy(
    "warehouse_id"
).agg(
    spark_sum("current_stock").alias("total_stock"),
    spark_sum(
        when(col("stock_status") == "Understocked", 1)
        .otherwise(0)
    ).alias("understocked_items"),
    spark_sum(
        when(col("stock_status") == "Overstocked", 1)
        .otherwise(0)
    ).alias("overstocked_items")
)

print("Warehouse-level summary:")
warehouse_summary.show()
warehouse_summary.coalesce(1).write.mode(
    "overwrite"
).option("header", True).csv(
    "warehouse_summary_output"
)
stock.coalesce(1).write.mode(
    "overwrite"
).option("header", True).csv(
    "product_stock_status_output"
)

print("Inventory analysis completed!")
spark.stop()